In [1]:
from pathlib import Path
import os
import sys

import mne
import numpy as np

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)

os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%matplotlib qt

In [2]:
from modules.decode_trigger import decode_8bit_trigger, convert_dict_trigger
from modules.events import get_events_tms_per_task
from modules.ica import ICAProcessor

c:\Repositorios\msc-eeg-tms-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
raw_data = mne.io.read_raw_bdf(r"data/raw/V1.bdf", preload=True)

Extracting BDF parameters from data/raw/V1.bdf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 12594999  =      0.000 ...  2519.000 secs...


In [ ]:
emg_ch_names = ["EMG L", "EMG R"]
eog_ch_names = ["EOG"]
raw_data.set_channel_types({ch: "emg" for ch in emg_ch_names})
raw_data.set_channel_types({'EOG':'eog'})

montage = mne.channels.make_standard_montage("standard_1020",head_size='auto')
raw_data.set_montage(montage)

In [ ]:
raw_data.drop_channels(emg_ch_names)

In [ ]:
bad_ch =['F7', 'F5', 'P7', 'CPz', 'Oz', 'Iz', 'PO4', 'PO8', 'O2', "P6", "P8", "TP8", "C6", "TP10", "TP9", "F6", "FT8", "T8", "T7", "F3", "FC5", "FC3", "C3", "C1"]
raw_data.drop_channels(bad_ch)

In [7]:
events, event_id = mne.events_from_annotations(raw_data)
event_id = raw_data.event_id = convert_dict_trigger(event_id, decode_8bit_trigger)

Used Annotations descriptions: [np.str_('8Bit 1'), np.str_('8Bit 10'), np.str_('8Bit 2'), np.str_('8Bit 3'), np.str_('8Bit 4'), np.str_('8Bit 5'), np.str_('Stimulus A')]


In [8]:
raw_data = mne.preprocessing.fix_stim_artifact(
    raw_data, 
    events=events,
    event_id=7,
    tmin=-0.004,  # -2 ms
    tmax=0.018, # 10 ms
    mode='linear'
)

In [9]:
filtered_data = raw_data.copy().filter(l_freq=1, h_freq=100, method='iir', iir_params={'order': 4, 'ftype': 'butter'})

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 1e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 1.00, 100.00 Hz: -6.02, -6.02 dB



In [ ]:
freqs_notch = np.arange(60, 241, 60)
filtered_data.notch_filter(freqs=freqs_notch, picks='eeg')

In [ ]:
## Cálculo do Rank Efetivo

In [11]:
events_tms, events_id_tms = get_events_tms_per_task(events, event_id)

In [18]:
epochs = mne.Epochs(
    filtered_data,
    events=events_tms,
    event_id=events_id_tms,
    tmin=-2,
    tmax=0.05,
    preload=True,
)

Not setting metadata
100 matching events found
Setting baseline interval to [-2.0, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 100 events and 10251 original time points ...
0 bad epochs dropped


In [ ]:
epochs.plot()

In [20]:
ica = ICAProcessor()

In [21]:
ica.fit(epochs)

Fitting ICA to data using 40 channels (please be patient, this may take a while)


c:\Repositorios\msc-eeg-tms-pipeline\modules\ica.py:10: RuntimeWarning: The epochs you passed to ICA.fit() were baseline-corrected. However, we suggest to fit ICA only on data that has been high-pass filtered, but NOT baseline-corrected.
  self.ica.fit(epoch)


Selecting by non-zero PCA components: 40 components
Computing Infomax ICA
Fitting ICA took 358.1s.


In [22]:
ica.plot_components()

Not setting metadata
100 matching events found
No baseline correction applied
0 projection items activated


In [ ]:
exclide_comp = ica.get_exclude_components()